# 01 – RQ1: Chi-Square Test of Independence

**Research Question:** Do enterprise-deployed AI incidents exhibit significantly different risk domain distributions compared to non-enterprise incidents?

**Hypothesis:** Enterprise incidents are disproportionately concentrated in 'AI system safety, failures, and limitations', while non-enterprise incidents dominate 'Malicious Actors and Misuse'.

**Method:** Chi-square test of independence. Fisher's exact test if >20% of expected cell counts fall below 5. Effect size reported as Cramer's V.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

FIGURES_DIR = Path('../outputs/figures')
df = pd.read_pickle('../outputs/incidents_prepared.pkl')

In [ ]:
# Identify Risk Domain column
domain_col = [c for c in df.columns if 'risk domain' in c.lower() or 'riskofdomain' in c.lower()]
print('Risk Domain column candidates:', domain_col)

# Work with the first candidate; adjust if needed
DOMAIN_COL = domain_col[0] if domain_col else None
print('Using column:', DOMAIN_COL)

In [ ]:
# Subset to records where Risk Domain is populated
rq1_df = df[df[DOMAIN_COL].notna() & df['Deployer_Type'].notna()].copy()
print(f'Records for RQ1 analysis: {len(rq1_df):,}')
print('Risk Domain distribution:')
print(rq1_df[DOMAIN_COL].value_counts())

In [ ]:
# Build contingency table
ct = pd.crosstab(rq1_df['Deployer_Type'], rq1_df[DOMAIN_COL])
print('Contingency Table (Observed Frequencies):')
ct

In [ ]:
# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(ct)

# Check assumption: proportion of expected cells < 5
low_expected = (expected < 5).sum() / expected.size
print(f'Chi-square statistic: {chi2:.4f}')
print(f'Degrees of freedom:   {dof}')
print(f'p-value:              {p:.6f}')
print(f'Expected cells < 5:   {low_expected:.1%}')

if low_expected > 0.20:
    print('WARNING: >20% of expected cells are below 5. Use Fisher\'s exact test or collapse categories.')
else:
    print('Assumption met: chi-square results are valid.')

In [ ]:
# Effect size: Cramer's V
n = ct.values.sum()
min_dim = min(ct.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))
print(f"Cramer's V: {cramers_v:.4f}")

# Interpretation thresholds (Cohen, 1988)
if cramers_v < 0.10:
    strength = 'negligible'
elif cramers_v < 0.30:
    strength = 'small'
elif cramers_v < 0.50:
    strength = 'medium'
else:
    strength = 'large'
print(f'Effect size interpretation: {strength}')

In [ ]:
# Visualisation: stacked bar chart of risk domain proportions by deployer type
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 5))
ct_pct.T.plot(kind='bar', ax=ax)
ax.set_title('Risk Domain Distribution by Deployer Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Risk Domain')
ax.set_ylabel('Percentage of Incidents (%)')
ax.legend(title='Deployer Type', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rq1_risk_domain_by_deployer_type.png', dpi=150)
plt.show()